# TunBERT Baseline — Topic Modeling of Tunisian Dialect

**Method:** Contextual embeddings from **TunBERT** (a BERT model pretrained on Tunisian dialect) feed a bag-of-words vectorizer and coherence-scoring utilities used as the baseline for the rest of the study.

**Why this method:** TunBERT is, to date, the only widely available transformer pretrained specifically on Tunisian dialect (as opposed to Modern Standard Arabic or generic multilingual models). It's the natural starting point / baseline against which the hybrid embedding combinations in the other notebooks are compared.

**What this notebook produces:** document embeddings from TunBERT, a bag-of-words representation, and the coherence-scoring utilities (`compute_cv_score`, `compute_manual_npmi`) that every other notebook in this project reuses.

## 0. Setup

In [ ]:
!pip install transformers sentence-transformers scikit-learn nltk pandas openpyxl gensim


## 1. Load the corpus

We start from the raw Tunisian dialect social media corpus. Each row is one post/message; we drop empty rows since they carry no signal for topic modeling.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel("../data/TunTap_Corpus.xlsx")

# Drop rows with missing text
df = df.dropna(subset=["message"])

# Reset index
df = df.reset_index(drop=True)

# Extract just the text column
message = df["message"].tolist()


## 2. Preprocessing

Tunisian dialect on social media mixes Arabic script, Arabizi (Latin transliteration with digits standing in for Arabic letters, e.g. `3` for `ع`), French loanwords, elongated words for emphasis (`hhhhh`, `mriiiigla`), and noise (URLs, mentions, hashtags, emojis). Standard NLP preprocessing pipelines aren't built for this, so we apply dialect-specific cleaning:

1. Remove custom Tunisian stopwords (functional words with no topical meaning)
2. Normalize character elongation (`mriiiigla` → `mrigla`)
3. Convert Arabizi digits back to their Arabic-letter equivalent (`3` → `a`, `7` → `h`, `5` → `kh`, `9` → `k`, `2` → `a`)
4. Strip URLs, mentions, hashtags, emojis and non-alphanumeric symbols
5. Normalize Arabic letter variants (e.g. `إأآا` → `ا`) so the same word isn't split across multiple spellings
6. Remove stopwords a second time (some appear only after cleaning) and drop any resulting empty lines

In [ ]:
# Read stopwords.txt into a set
with open("../data/tunisian_stopwords.txt", "r", encoding="utf-8") as f:
    all_stopwords = set(line.strip() for line in f if line.strip())


In [ ]:
def remove_custom_stopwords(text, stopwords_set):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stopwords_set]
    return " ".join(filtered)


In [ ]:
text = [remove_custom_stopwords(t, all_stopwords) for t in message]


In [ ]:
import re

def normalize_elongation(text):
    # Replace 2 or more repeated characters with 1
    return re.sub(r'(.)\1{1,}', r'\1', text)

def convert_tunisian_numbers(text):
    return (
        text.replace("3", "a")
            .replace("7", "h")
            .replace("5", "kh")
            .replace("9", "k")
            .replace("2", "a")
    )

def remove_numbers(text):
    # Remove all digits (0-9)
    return re.sub(r'\d+', '', text)

def clean_dual_script_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)

    # Remove emojis and symbols (non-alphanum + Arabic letters + space)
    text = re.sub(r"[^\u0621-\u063A\u0641-\u064A\w\s]", " ", text)

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # convert numbers like 3 → ع
    text = convert_tunisian_numbers(text)

    # Normalize elongation
    text = normalize_elongation(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    text = remove_numbers(text)

    return text


In [ ]:
cleaned_texts = [clean_dual_script_text(t) for t in text]

In [ ]:
final_texts = [remove_custom_stopwords(t, all_stopwords) for t in cleaned_texts]

In [ ]:
final_texts = [line for line in final_texts if line.strip() != '']


Check how many documents survived preprocessing:

In [ ]:
print(len(final_texts))


17287


## 4. Load TunBERT and generate document embeddings

We load TunBERT via its Hugging Face `not-lain/TunBERT` checkpoint and encode each preprocessed document by taking the `[CLS]` token's last hidden state — the standard way to get a single fixed-size vector per document from a BERT-style model.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# Load TunBERT model (classifier version)
tokenizer = AutoTokenizer.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model.eval()


In [ ]:
def embed_documents(docs, batch_size=32):
    embeddings = []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    with torch.no_grad():
        for i in range(0, len(docs), batch_size):
            batch = docs[i:i+batch_size]
            for doc in batch:
                inputs = tokenizer(doc, return_tensors="pt", truncation=True, padding=True).to(device)
                outputs = model.BertModel(**inputs, output_hidden_states=True)
                cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token embedding
                embeddings.append(cls_embedding.squeeze().cpu().numpy())
    return np.array(embeddings)


In [ ]:
print("Embedding documents with TunBERT...")
embeddings = embed_documents(final_texts)
print("Done embeddings. Shape:", embeddings.shape)


Embedding documents with TunBERT...
Done embeddings. Shape: (17287, 768)


## 5. Bag-of-words representation

CombinedTM (used throughout this project) needs both a contextual embedding **and** a bag-of-words vector per document — the BoW half is what lets the model reconstruct interpretable word-level topics. We build it here with unigrams and bigrams, using the same Tunisian stopword list.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import torch
import umap

vectorizer_model = CountVectorizer(
    stop_words=list(all_stopwords),
    tokenizer=lambda x: x.split(),
    ngram_range=(1, 2)
)

vectorizer_model.fit(final_texts)
X_bow = vectorizer_model.transform(final_texts)  # sparse matrix


## 3. Topic coherence utilities

We evaluate topic quality with two complementary metrics:
- **C_V coherence** — measures how semantically related the top words of a topic are, based on word co-occurrence in a sliding window (via `gensim`)
- **NPMI** (Normalized Pointwise Mutual Information) — a simpler, more interpretable co-occurrence measure computed directly on document sets, less sensitive to corpus size than raw PMI

Both are computed from `final_texts`, split into tokens.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from itertools import combinations
from collections import Counter
import math

tokenized_docs = [doc.split() for doc in final_texts]
dictionary = Dictionary(tokenized_docs)

def compute_cv_score(topics_list, tokenized_docs, dictionary):
    cm = CoherenceModel(topics=topics_list, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def compute_manual_npmi(topics_list, tokenized_docs):
    doc_sets = [set(d) for d in tokenized_docs]
    num_docs = len(doc_sets)
    word_doc_counts = Counter()
    for s in doc_sets:
        for w in s:
            word_doc_counts[w] += 1
    def topic_npmi(topic_words):
        vals = []
        for w1, w2 in combinations(topic_words, 2):
            p1 = word_doc_counts.get(w1, 0) / num_docs
            p2 = word_doc_counts.get(w2, 0) / num_docs
            co = sum(1 for s in doc_sets if w1 in s and w2 in s)
            p12 = co / num_docs
            if p12 > 0 and p1 > 0 and p2 > 0:
                pmi = math.log(p12 / (p1 * p2))
                npmi = pmi / (-math.log(p12))
                vals.append(npmi)
        return (sum(vals) / len(vals)) if vals else 0.0
    scores = [topic_npmi(topic) for topic in topics_list]
    return sum(scores) / len(scores), scores


## Notes

- `embeddings` (TunBERT, shape `(n_docs, 768)`), `X_bow`, and the coherence utilities defined here are the building blocks reused across all the other notebooks in this project (each notebook regenerates them independently since it's self-contained).
- This notebook doesn't train a full CombinedTM by itself — it establishes the baseline embeddings and evaluation tools. See the other notebooks for the actual topic model training and results per embedding combination.